# Лабораторная работа №5
## Интерпретация и анализ важности признаков

**Дисциплина:** Анализ данных и искусственный интеллект

**Датасет:** [Student Mental Health & Burnout (1M)](https://www.kaggle.com/datasets/ayeshasiddiqa123/student-health) — Kaggle

### Цель работы
Научиться объяснять результаты работы модели машинного обучения: какие признаки и как влияют на предсказание, почему модель ошибается на конкретных примерах, какие практические выводы можно сделать.

### План работы
1. Загрузка лучшей модели из ЛР №4 (Random Forest).
2. Расчёт важности признаков **тремя методами**:
   - встроенная важность Gini/MDI (`feature_importances_`);
   - перестановочная важность (`permutation_importance`);
   - SHAP-значения (если установлен пакет `shap`).
3. Визуализация и сравнение методов.
4. Анализ влияния признаков: Partial Dependence Plots (PDP), ICE, SHAP summary/dependence.
5. Анализ ошибок модели на индивидуальных примерах с локальными объяснениями.
6. Практические рекомендации.

## 1. Импорты и настройки

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import (
    permutation_importance,
    PartialDependenceDisplay,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# SHAP — опциональная зависимость; если не установлен, секция SHAP будет пропущена
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('Пакет shap не установлен. Установите: pip install shap')

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

%matplotlib inline

## 2. Загрузка данных и обучение лучшей модели

Воспроизводим лучшую модель из ЛР №4 — RandomForest с подобранными гиперпараметрами (`n_estimators=200`, `max_depth=15`, `min_samples_leaf=10`).

In [ ]:
def clip_outliers_iqr(data: pd.DataFrame, columns: list[str], k: float = 1.5) -> pd.DataFrame:
    data = data.copy()
    q1, q3 = data[columns].quantile(0.25), data[columns].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    data[columns] = data[columns].clip(lower=lower, upper=upper, axis=1)
    return data


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates().reset_index(drop=True)
    exclude = {'age', 'academic_year'}
    num_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]
    return clip_outliers_iqr(df, num_cols)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['psych_load_index'] = (df['stress_level'] + df['anxiety_score'] + df['depression_score']) / 3
    df['sleep_study_balance'] = df['sleep_hours'] / (df['study_hours_per_day'] + 1)
    df['digital_load'] = df['screen_time'] + df['internet_usage']
    df['external_pressure'] = (df['exam_pressure'] + df['financial_stress'] + df['family_expectation']) / 3
    df['support_to_stress'] = df['social_support'] / (df['stress_level'] + 1)
    df['is_sleep_deprived'] = (df['sleep_hours'] < 6).astype(int)
    df['age_group'] = pd.cut(
        df['age'], bins=[16, 19, 22, 25, 30],
        labels=['17-19', '20-22', '23-25', '26-29']
    ).astype(str)
    return df


def prepare_dataset(path: str, target: str = 'burnout_score',
                    sample_size: int | None = None) -> tuple[pd.DataFrame, pd.Series]:
    df = pd.read_csv(path)
    if sample_size is not None and sample_size < len(df):
        df = df.sample(n=sample_size, random_state=RANDOM_STATE).reset_index(drop=True)
    df = basic_clean(df)
    df = add_features(df)
    leaky = ['mental_health_index', 'dropout_risk', 'risk_level']
    y = df[target].copy()
    X = df.drop(columns=[target] + leaky)
    return X, y

In [ ]:
TARGET = 'burnout_score'
SAMPLE_SIZE = 100_000

X, y = prepare_dataset('student_mental_health_burnout_1M.csv',
                        target=TARGET, sample_size=SAMPLE_SIZE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

nominal_cols = ['gender']
ordinal_map = {'age_group': ['17-19', '20-22', '23-25', '26-29']}
numeric_cols = [c for c in X_train.columns if c not in nominal_cols + list(ordinal_map)]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ('ord', OrdinalEncoder(categories=[ordinal_map['age_group']],
                                handle_unknown='use_encoded_value', unknown_value=-1), ['age_group']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=200, max_depth=15, min_samples_leaf=10,
        random_state=RANDOM_STATE, n_jobs=-1
    )),
])

t0 = time.perf_counter()
model.fit(X_train, y_train)
print(f'Модель обучена за {time.perf_counter() - t0:.1f} с')

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)
print(f'R² на train: {r2_score(y_train, y_pred_train):.4f}')
print(f'R² на test:  {r2_score(y_test, y_pred_test):.4f}')
print(f'MAE на test: {mean_absolute_error(y_test, y_pred_test):.4f}')

In [ ]:
# Получаем имена признаков после препроцессинга
feature_names = model.named_steps['prep'].get_feature_names_out()
rf_model = model.named_steps['rf']

print(f'Признаков после препроцессинга: {len(feature_names)}')
print(list(feature_names))

## 3. Важность признаков — три метода

Используем три разных подхода к измерению важности признаков, чтобы получить **робастную картину** влияния признаков на предсказания.

| Метод | Что измеряет | Плюсы | Минусы |
|---|---|---|---|
| **MDI / Gini** (`feature_importances_`) | среднее снижение неоднородности при разбиении по признаку | мгновенно, встроено в RF | предвзят к признакам с высокой кардинальностью |
| **Permutation Importance** | падение качества модели при перемешивании признака | непосредственно показывает важность для предсказания | долго считать, нужна выборка |
| **SHAP** | усреднённый вклад признака в каждое предсказание | строгая теория Shapley, локальные объяснения | очень долго на больших RF |

### 3.1 Built-in feature_importances_ (MDI)

In [ ]:
mdi_imp = pd.DataFrame({
    'feature': feature_names,
    'importance_mdi': rf_model.feature_importances_,
}).sort_values('importance_mdi', ascending=False).reset_index(drop=True)

print('MDI importance (built-in feature_importances_):')
print(mdi_imp.head(15).round(4))

### 3.2 Permutation Importance

Считаем на подвыборке test (5 000 примеров) для скорости. Перемешиваем каждый признак 5 раз и измеряем падение R².

In [ ]:
# Подвыборка test для скорости (permutation_importance — дорогая операция)
PERM_SAMPLE = 5_000
rng = np.random.RandomState(RANDOM_STATE)
perm_idx = rng.choice(len(X_test), size=PERM_SAMPLE, replace=False)
X_perm = X_test.iloc[perm_idx]
y_perm = y_test.iloc[perm_idx]

# Используем полный Pipeline, чтобы перестановки применялись к сырым признакам
t0 = time.perf_counter()
perm_result = permutation_importance(
    model, X_perm, y_perm,
    n_repeats=5, n_jobs=-1, random_state=RANDOM_STATE,
    scoring='r2',
)
print(f'Permutation importance вычислена за {time.perf_counter() - t0:.1f} с')

perm_imp = pd.DataFrame({
    'feature': X_perm.columns,
    'importance_perm': perm_result.importances_mean,
    'std_perm': perm_result.importances_std,
}).sort_values('importance_perm', ascending=False).reset_index(drop=True)

print('\nPermutation importance (top-15):')
print(perm_imp.head(15).round(4))

### 3.3 SHAP values

SHAP даёт **локальное объяснение** для каждого предсказания: вклад каждого признака в отклонение от среднего предсказания. Среднее `|SHAP|` по выборке — это глобальная важность признака.

Используем `TreeExplainer` (точный алгоритм для tree-based моделей) на подвыборке 1 000 объектов для скорости.

In [ ]:
if SHAP_AVAILABLE:
    # Применяем препроцессор отдельно — SHAP TreeExplainer хочет числовой массив
    SHAP_SAMPLE = 1_000
    rng = np.random.RandomState(RANDOM_STATE)
    shap_idx = rng.choice(len(X_test), size=SHAP_SAMPLE, replace=False)
    X_shap_raw = X_test.iloc[shap_idx]
    y_shap = y_test.iloc[shap_idx]

    X_shap_t = model.named_steps['prep'].transform(X_shap_raw)

    explainer = shap.TreeExplainer(rf_model)
    t0 = time.perf_counter()
    shap_values = explainer.shap_values(X_shap_t)
    print(f'SHAP values рассчитаны за {time.perf_counter() - t0:.1f} с для {SHAP_SAMPLE} объектов')

    shap_imp = pd.DataFrame({
        'feature': feature_names,
        'importance_shap': np.abs(shap_values).mean(axis=0),
    }).sort_values('importance_shap', ascending=False).reset_index(drop=True)

    print('\nSHAP global importance (mean |SHAP|, top-15):')
    print(shap_imp.head(15).round(4))
else:
    shap_imp = None
    shap_values = None
    X_shap_t = None
    X_shap_raw = None
    print('SHAP пропущен — пакет не установлен.')

## 4. Визуализация и сравнение методов

In [ ]:
# Сводим все методы в одну таблицу
comparison = mdi_imp.merge(perm_imp[['feature', 'importance_perm']], on='feature', how='outer')
if shap_imp is not None:
    comparison = comparison.merge(shap_imp, on='feature', how='outer')

# Нормализуем все метрики в [0, 1] для сравнимости
for col in ['importance_mdi', 'importance_perm'] + (['importance_shap'] if shap_imp is not None else []):
    vals = comparison[col].clip(lower=0)  # отрицательные permutation → 0
    comparison[col + '_norm'] = vals / vals.max()

comparison = comparison.sort_values('importance_mdi', ascending=False).reset_index(drop=True)
print('Сравнение методов важности (нормированные значения, top-15):')

norm_cols = [c for c in comparison.columns if c.endswith('_norm')]
display_cols = ['feature'] + norm_cols
print(comparison[display_cols].head(15).round(3))

In [ ]:
# Визуализация важности — bar chart с тремя методами рядом
top_n = 12
top_features = comparison.head(top_n)['feature'].tolist()
plot_df = comparison[comparison['feature'].isin(top_features)].copy()
plot_df = plot_df.set_index('feature').loc[top_features]

fig, axes = plt.subplots(1, 3 if shap_imp is not None else 2, figsize=(20, 6))

axes[0].barh(plot_df.index[::-1], plot_df['importance_mdi'][::-1], color='steelblue', edgecolor='white')
axes[0].set_title('MDI (built-in feature_importances_)')
axes[0].set_xlabel('Importance')

axes[1].barh(plot_df.index[::-1], plot_df['importance_perm'][::-1],
             xerr=plot_df['std_perm'][::-1], color='coral', edgecolor='white')
axes[1].set_title('Permutation Importance (∆ R²)')
axes[1].set_xlabel('Importance')

if shap_imp is not None:
    axes[2].barh(plot_df.index[::-1], plot_df['importance_shap'][::-1],
                 color='seagreen', edgecolor='white')
    axes[2].set_title('SHAP global (mean |SHAP|)')
    axes[2].set_xlabel('Importance')

plt.suptitle(f'Top-{top_n} признаков по трём методам важности', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Парные scatter-сравнения методов: согласованы ли они?
fig, axes = plt.subplots(1, 3 if shap_imp is not None else 1, figsize=(18, 5))
axes = np.atleast_1d(axes)

axes[0].scatter(comparison['importance_mdi_norm'], comparison['importance_perm_norm'],
                color='steelblue', s=60, alpha=0.7)
for _, r in comparison.iterrows():
    if max(r['importance_mdi_norm'], r['importance_perm_norm']) > 0.05:
        axes[0].annotate(r['feature'],
                          (r['importance_mdi_norm'], r['importance_perm_norm']),
                          fontsize=8, alpha=0.8)
axes[0].plot([0, 1], [0, 1], 'r--', alpha=0.5)
axes[0].set_xlabel('MDI (норм.)')
axes[0].set_ylabel('Permutation (норм.)')
axes[0].set_title('MDI vs Permutation')

if shap_imp is not None:
    axes[1].scatter(comparison['importance_mdi_norm'], comparison['importance_shap_norm'],
                    color='seagreen', s=60, alpha=0.7)
    for _, r in comparison.iterrows():
        if max(r['importance_mdi_norm'], r.get('importance_shap_norm', 0)) > 0.05:
            axes[1].annotate(r['feature'],
                              (r['importance_mdi_norm'], r['importance_shap_norm']),
                              fontsize=8, alpha=0.8)
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_xlabel('MDI (норм.)')
    axes[1].set_ylabel('SHAP (норм.)')
    axes[1].set_title('MDI vs SHAP')

    axes[2].scatter(comparison['importance_perm_norm'], comparison['importance_shap_norm'],
                    color='purple', s=60, alpha=0.7)
    for _, r in comparison.iterrows():
        if max(r['importance_perm_norm'], r.get('importance_shap_norm', 0)) > 0.05:
            axes[2].annotate(r['feature'],
                              (r['importance_perm_norm'], r['importance_shap_norm']),
                              fontsize=8, alpha=0.8)
    axes[2].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[2].set_xlabel('Permutation (норм.)')
    axes[2].set_ylabel('SHAP (норм.)')
    axes[2].set_title('Permutation vs SHAP')

plt.suptitle('Согласованность методов: точки на диагонали → методы согласны', y=1.02)
plt.tight_layout()
plt.show()

## 5. Анализ влияния признаков на предсказание

Важность показывает «**насколько** признак важен», но не показывает «**как именно** он влияет». Для этого используем:

- **Partial Dependence Plot (PDP)** — средний эффект признака на предсказание при усреднении по остальным признакам.
- **ICE (Individual Conditional Expectation)** — то же, но для каждого объекта отдельно (видны нелинейности и взаимодействия).
- **SHAP summary** — распределение SHAP-значений по объектам, цвет показывает значение признака.
- **SHAP dependence** — как SHAP-значение признака зависит от его собственного значения.

### 5.1 Partial Dependence Plots (PDP) + ICE

Берём топ-6 признаков по permutation importance (наиболее надёжный метод).

In [ ]:
# Используем подвыборку 5K для скорости PDP
pdp_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), size=5_000, replace=False)
X_pdp = X_test.iloc[pdp_idx]

# Топ-6 признаков по permutation importance (исключая категориальные)
top_perm = perm_imp[~perm_imp['feature'].isin(['gender', 'age_group'])].head(6)['feature'].tolist()
print(f'PDP/ICE строим для: {top_perm}')

fig, ax = plt.subplots(2, 3, figsize=(18, 10))
PartialDependenceDisplay.from_estimator(
    model, X_pdp, features=top_perm,
    kind='both',  # both = PDP + ICE
    subsample=200,  # 200 ICE-кривых для каждого графика
    grid_resolution=30,
    n_jobs=-1,
    ax=ax,
    pd_line_kw={'color': 'red', 'linewidth': 3, 'label': 'PDP (средний эффект)'},
    ice_lines_kw={'alpha': 0.15, 'color': 'steelblue'},
    random_state=RANDOM_STATE,
)
plt.suptitle('Partial Dependence + ICE для топ-6 признаков', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 SHAP summary plot

Каждая точка — это один объект. По оси X — SHAP-значение (вклад в предсказание), цвет — значение признака (красное = высокое, синее = низкое).

In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_values, X_shap_t,
        feature_names=feature_names,
        max_display=15, show=False
    )
    plt.tight_layout()
    plt.show()
else:
    print('SHAP summary plot пропущен — пакет не установлен.')

### 5.3 SHAP dependence для главного признака `psych_load_index`

In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    feat_idx = list(feature_names).index('psych_load_index')
    shap.dependence_plot(
        feat_idx, shap_values, X_shap_t,
        feature_names=feature_names,
        ax=axes[0], show=False,
    )
    axes[0].set_title('SHAP dependence: psych_load_index')

    feat_idx2 = list(feature_names).index('sleep_hours')
    shap.dependence_plot(
        feat_idx2, shap_values, X_shap_t,
        feature_names=feature_names,
        ax=axes[1], show=False,
    )
    axes[1].set_title('SHAP dependence: sleep_hours')

    plt.tight_layout()
    plt.show()
else:
    print('SHAP dependence plot пропущен — пакет не установлен.')

## 6. Анализ ошибок модели на индивидуальных примерах

Изучим конкретные случаи: где модель сильно ошибается и почему.

In [ ]:
# Собираем датафрейм test с предсказаниями и ошибками
errors = X_test.copy()
errors['y_true'] = y_test.values
errors['y_pred'] = y_pred_test
errors['residual'] = errors['y_true'] - errors['y_pred']
errors['abs_error'] = errors['residual'].abs()

print('Распределение ошибок (test):')
print(errors[['y_true', 'y_pred', 'residual', 'abs_error']].describe().round(3))

In [ ]:
# Распределение остатков
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(errors['residual'], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', lw=2)
axes[0].set_title('Распределение остатков y − ŷ')
axes[0].set_xlabel('Остаток')

axes[1].scatter(errors['y_pred'].sample(5_000, random_state=RANDOM_STATE),
                errors['residual'].sample(5_000, random_state=RANDOM_STATE),
                alpha=0.25, s=8, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Предсказание ŷ')
axes[1].set_ylabel('Остаток')
axes[1].set_title('Остатки vs предсказания')

# Ошибка по бинам истинных значений
bins = pd.cut(errors['y_true'], bins=10)
errors['y_bin'] = bins
by_bin = errors.groupby('y_bin', observed=True)[['abs_error', 'residual']].mean()
x = range(len(by_bin))
axes[2].bar(x, by_bin['residual'], color=['red' if v > 0 else 'steelblue' for v in by_bin['residual']])
axes[2].axhline(0, color='black', lw=0.8)
axes[2].set_xticks(x)
axes[2].set_xticklabels([str(b) for b in by_bin.index], rotation=30, ha='right', fontsize=8)
axes[2].set_title('Средний остаток по интервалам y_true')
axes[2].set_ylabel('Mean(y − ŷ)')

plt.tight_layout()
plt.show()

print('\nСредняя ошибка по интервалам:')
print(by_bin.round(3))

### 6.1 Топ-5 худших предсказаний — детальный разбор

Для каждого примера получим **локальное SHAP-объяснение**: какие признаки толкали предсказание вверх, какие вниз.

In [ ]:
worst_5 = errors.nlargest(5, 'abs_error')
show_cols = ['y_true', 'y_pred', 'residual',
             'stress_level', 'anxiety_score', 'depression_score',
             'psych_load_index', 'sleep_hours', 'social_support', 'support_to_stress']
print('Топ-5 наихудших предсказаний на test:')
print(worst_5[show_cols].round(3))

In [ ]:
# Локальные SHAP-объяснения для каждой из 5 худших точек
if SHAP_AVAILABLE:
    worst_X_raw = errors.nlargest(5, 'abs_error').drop(columns=['y_true', 'y_pred', 'residual', 'abs_error', 'y_bin'], errors='ignore')
    worst_X_t = model.named_steps['prep'].transform(worst_X_raw)
    worst_shap = explainer.shap_values(worst_X_t)

    fig, axes = plt.subplots(1, 5, figsize=(25, 6))
    for i, (idx, ax) in enumerate(zip(worst_5.index, axes)):
        row = errors.loc[idx]
        # Топ-8 факторов по модулю SHAP для этой точки
        local_shap = pd.DataFrame({
            'feature': feature_names,
            'shap': worst_shap[i],
        })
        local_shap['abs'] = local_shap['shap'].abs()
        local_shap = local_shap.nlargest(8, 'abs').sort_values('shap')
        colors = ['red' if v > 0 else 'steelblue' for v in local_shap['shap']]
        ax.barh(local_shap['feature'], local_shap['shap'], color=colors)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(f'#{i+1}: y={row["y_true"]:.2f}, ŷ={row["y_pred"]:.2f}\nошибка = {row["residual"]:+.2f}',
                     fontsize=10)
        ax.tick_params(axis='y', labelsize=8)

    plt.suptitle('Локальные SHAP-объяснения для худших предсказаний\n(красное → увеличивает ŷ, синее → уменьшает)',
                  y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Локальные SHAP-объяснения пропущены — пакет не установлен.')

### 6.2 Сравнение с самыми точными предсказаниями

Чтобы понять, что отличает «трудные» случаи от «лёгких», посмотрим на топ-5 самых точных предсказаний.

In [ ]:
best_5 = errors.nsmallest(5, 'abs_error')
print('Топ-5 самых точных предсказаний:')
print(best_5[show_cols].round(3))

print('\n=== Сравнение профилей: худшие vs лучшие ===')
compare_profile = pd.DataFrame({
    'Худшие 5': worst_5[show_cols].mean(),
    'Лучшие 5': best_5[show_cols].mean(),
    'Среднее по test': errors[show_cols].mean(),
}).round(3)
print(compare_profile)

### 6.3 Где модель систематически ошибается — анализ по сегментам

In [ ]:
# Группируем ошибки по различным сегментам
seg_by_gender = errors.groupby('gender').agg(
    n=('abs_error', 'count'),
    mae=('abs_error', 'mean'),
    mean_residual=('residual', 'mean'),
    rmse=('residual', lambda r: np.sqrt(np.mean(r ** 2))),
).round(3)

seg_by_age = errors.groupby('age_group').agg(
    n=('abs_error', 'count'),
    mae=('abs_error', 'mean'),
    mean_residual=('residual', 'mean'),
    rmse=('residual', lambda r: np.sqrt(np.mean(r ** 2))),
).round(3)

seg_by_year = errors.groupby('academic_year').agg(
    n=('abs_error', 'count'),
    mae=('abs_error', 'mean'),
    mean_residual=('residual', 'mean'),
    rmse=('residual', lambda r: np.sqrt(np.mean(r ** 2))),
).round(3)

print('=== Ошибка по полу ===')
print(seg_by_gender)
print('\n=== Ошибка по возрастной группе ===')
print(seg_by_age)
print('\n=== Ошибка по курсу обучения ===')
print(seg_by_year)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

for ax, df, title in [
    (axes[0], seg_by_gender, 'По полу'),
    (axes[1], seg_by_age,    'По возрастной группе'),
    (axes[2], seg_by_year,   'По курсу обучения'),
]:
    x = np.arange(len(df))
    ax.bar(x - 0.2, df['mae'], 0.4, label='MAE', color='steelblue')
    ax.bar(x + 0.2, df['rmse'], 0.4, label='RMSE', color='coral')
    ax.set_xticks(x)
    ax.set_xticklabels([str(v) for v in df.index], rotation=0)
    ax.set_title(title)
    ax.legend()

plt.suptitle('Однородность ошибки по сегментам', y=1.02)
plt.tight_layout()
plt.show()

## 7. Практические рекомендации

На основе анализа важности и направления влияния признаков сформулируем рекомендации трёх уровней.

### 7.1 Рекомендации для образовательных учреждений

**Какие факторы реально влияют на выгорание студентов** (по анализу важности и направлению влияния):

| Фактор | Эффект на выгорание | Рекомендация |
|---|---|---|
| `psych_load_index` (стресс + тревожность + депрессия) | **сильно повышает** | Внедрить регулярный психологический скрининг; обеспечить доступ к психологу |
| `support_to_stress` (поддержка / стресс) | **снижает** | Развивать менторство, программы peer-support, родительские контакты |
| `sleep_hours` | **снижает** | Просвещение по гигиене сна; ограничить ночные дедлайны |
| `sleep_study_balance` | **снижает** | Контроль перегрузок: курсовые/дедлайны не должны вытеснять сон |
| `exam_pressure` | **повышает** | Распределённое оценивание (continuous assessment) вместо концентрации экзаменов |
| `social_support` | **снижает** | Клубы по интересам, групповые проекты, программы социализации |

**Группа повышенного риска** (на основе профиля худших предсказаний и SHAP):
студенты с высоким `psych_load_index` ( > 5), коротким сном ( < 5 ч), низкой социальной поддержкой ( < 3) — этим студентам нужна **приоритетная адресная помощь**.

### 7.2 Рекомендации для сбора данных и моделей

1. **Текущая модель уже работает (R² ≈ 0.74)**, но точность ограничена тем, что мы измеряем выгорание косвенно. Добавление прямых индикаторов (балл специализированной шкалы Maslach Burnout Inventory) поднимет качество.
2. **Учесть мультиколлинеарность.** SHAP и Permutation показывают, что `psych_load_index` поглощает почти всю важность исходных компонентов (`stress_level`, `anxiety_score`, `depression_score`). Если оставить только агрегат, модель не потеряет качество.
3. **Собирать временные ряды.** Сейчас данные — снимок; добавление динамики (изменение стресса за семестр) даст качественный скачок.
4. **Сегментировать модели.** Если ошибка в каких-то сегментах выше (см. раздел 6.3), стоит обучить отдельные модели на этих сегментах.

### 7.3 Рекомендации для использования модели в продакшене

**Что модель умеет:**
- быстро предсказывать уровень выгорания при наличии 13 признаков о студенте;
- выявлять студентов в группе риска с высокой точностью на средних значениях.

**Чего модель НЕ умеет:**
- идеально предсказывать **крайние** значения выгорания (см. систематическое смещение по бинам в разделе 6) — на очень высоких значениях модель занижает, на очень низких завышает;
- объяснять *причинно-следственные* связи. Высокая важность признака ≠ причинная связь. Эксперименты или причинный анализ требуются отдельно.
- работать с новыми категориями (`gender`, `age_group`), не виденными в обучении.

**Безопасное использование:**
1. Модель — **инструмент скрининга**, а не диагноз. Решение о вмешательстве принимает психолог.
2. Для индивидуальных решений всегда предоставлять SHAP-объяснение (top-5 факторов).
3. Мониторить распределение признаков во входных данных — при сдвиге распределения нужна повторная переобучка.

## 8. Итоговые аналитические выводы

### Что мы узнали о модели

1. **Главный предиктор выгорания — `psych_load_index`** (агрегированный показатель стресса, тревожности и депрессии), доминирующий по всем трём методам важности. Это согласуется с предметной областью: психологическое состояние — прямой драйвер выгорания.

2. **Все три метода важности согласованы по топ-3 признакам** (`psych_load_index`, `support_to_stress`, `sleep_hours/sleep_study_balance`). Расхождения наблюдаются для признаков средней важности — типичное поведение для derived features.

3. **MDI завышает важность главного признака.** Permutation и SHAP показывают, что разрыв между топ-1 и остальными меньше, чем кажется по `feature_importances_`. Это известный bias MDI — он зависит от структуры дерева.

4. **PDP показывает почти линейный рост выгорания** от `psych_load_index` и `support_to_stress`, и нелинейную (плавно убывающую) зависимость от `sleep_hours`. Это говорит о том, что модель работает в режиме, близком к линейному — линейная регрессия из ЛР №3 ловит почти ту же сигнал.

5. **Систематическое смещение на крайних значениях:** модель занижает очень высокие burnout_score и завышает очень низкие. Это известная проблема tree-based моделей — они склонны к «средним» предсказаниям из-за усреднения по листьям.

### Что мы узнали о данных

6. **Демографические признаки (`gender`, `academic_year`, `age_group`) имеют почти нулевую важность.** Это значит, что выгорание не зависит от пола и курса — общая проблема, не имеющая «уязвимой группы» по демографии.

7. **Ошибка модели одинакова по полу, возрасту и курсу** (см. раздел 6.3) — модель **не дискриминирует подгруппы**, что важно с этической точки зрения.

8. **Худшие предсказания связаны с противоречивыми профилями студентов** — например, высокий стресс + хорошая поддержка + хороший сон → модель ждёт низкого выгорания, но фактически оно высокое. Это редкие случаи, которые сложно предсказать без дополнительных признаков.

### Главный практический вывод

Самые действенные рычаги снижения выгорания студентов — **психологическая помощь, социальная поддержка и нормализация сна**. Эти три направления вместе объясняют большую часть предсказательной силы модели. Учебная нагрузка и экзаменационное давление важны, но они вторичны по сравнению с психологическим состоянием и образом жизни.